# 05. 뉴스/커뮤니티 데이터 수집

**저장 위치**: `data/raw/news_raw.csv` (15,211개)  
**목적**: FinBERT 감성 분석을 위한 텍스트 데이터 수집

| 소스 | 방법 | 수집량 | 특징 |
|------|------|--------|------|
| Reddit r/CryptoCurrency | Arctic Shift API (무료) | 12,987개 | 제목 + 본문 포함 |
| CoinTelegraph | 사이트맵 XML 파싱 | 2,224개 | 제목만 (본문 스크래핑 불가) |

## 수집 키워드
`USDT`, `USDC`, `DAI`, `stablecoin`, `stablecoin depeg`, `stablecoin depegging`

## 시도했으나 실패한 방법
- **CryptoPanic API**: 무료 플랜은 최근 20개만 제공 (역사 데이터 불가)
- **CoinDesk 검색 스크래핑**: JS 렌더링 방식이라 page 파라미터 효과 없음
- **CoinTelegraph 태그 페이지 offset 방식**: 동일 내용 반복 (JS 무한스크롤)
- **GDELT API**: 접속 타임아웃

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
import os
from datetime import datetime

SAVE_DIR   = os.path.join('..', 'data', 'raw')
START_DATE = '2019-11-22'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
}

# Reddit 수집 키워드 목록
REDDIT_KEYWORDS = ['USDT', 'USDC', 'DAI', 'stablecoin depeg', 'stablecoin depegging']

# CoinTelegraph URL에서 키워드 필터 (슬러그 기준)
CT_KEYWORDS = ['tether', 'usdt', 'usdc', 'usd-coin', 'dai', 'stablecoin', 'stable-coin', 'depeg', 'de-peg', 'peg']

# CoinTelegraph 사이트맵 XML 네임스페이스
NS = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

## Part 1. Reddit r/CryptoCurrency 수집

**Arctic Shift API**: Pushshift의 후속 서비스로 Reddit 역사 데이터 무료 제공  
- 엔드포인트: `https://arctic-shift.photon-reddit.com/api/posts/search`  
- 한 번에 최대 100개, `before` 파라미터로 페이지네이션

In [ ]:
def collect_reddit(keyword, after, before):
    """
    Arctic Shift API로 Reddit r/CryptoCurrency 게시물 수집
    - before: 이 날짜 이전 게시물 수집 (페이지네이션용)
    - after: 이 날짜 이후 게시물만 수집
    - limit=100: 한 번에 최대 100개
    - 마지막 게시물 날짜를 새 before로 설정해 반복
    """
    rows = []
    after_ts  = after
    before_ts = before

    while True:
        try:
            res = requests.get(
                'https://arctic-shift.photon-reddit.com/api/posts/search',
                params={
                    'subreddit': 'CryptoCurrency',
                    'query':     keyword,
                    'after':     after_ts,
                    'before':    before_ts,
                    'limit':     100,
                    'sort':      'desc'  # 최신순 정렬
                },
                timeout=20
            )
            if res.status_code != 200:
                break

            data = res.json().get('data', [])
            if not data:
                break

            for post in data:
                # created_utc: Unix 타임스탬프 → 날짜 변환
                created = datetime.utcfromtimestamp(post['created_utc']).strftime('%Y-%m-%d')
                rows.append({
                    'Date':   created,
                    'title':  post.get('title', ''),
                    'text':   post.get('selftext', ''),  # 본문 (링크 게시물은 빈값)
                    'source': 'reddit'
                })

            # 다음 페이지: 마지막 게시물 날짜를 before로 설정
            last_ts = data[-1]['created_utc']
            if last_ts <= 0 or len(data) < 100:
                break
            before_ts = str(datetime.utcfromtimestamp(last_ts).strftime('%Y-%m-%d'))
            time.sleep(1)

        except Exception as e:
            print(f'    Reddit 오류: {e}')
            break

    return rows

In [ ]:
# Reddit 수집 실행
# ⚠️ 키워드당 수백~수천 개 수집, 총 20~30분 소요 예상
reddit_rows = []

for kw in REDDIT_KEYWORDS:
    print(f'Reddit 키워드: {kw}')
    rows = collect_reddit(kw, START_DATE, END_DATE)
    reddit_rows.extend(rows)
    print(f'  → {len(rows)}개 수집')
    time.sleep(2)

print(f'\nReddit 총 수집: {len(reddit_rows)}개')

## Part 2. CoinTelegraph 수집 (사이트맵 방식)

**문제**: CoinTelegraph는 JS 무한스크롤 방식이라 일반 페이지네이션 불가  
**해결**: `sitemap.xml` 파일에는 모든 기사 URL과 수정 날짜가 정적으로 기재됨  
- 총 43개 사이트맵 파일, 각 1,000개 URL  
- URL 슬러그(제목-슬러그)에서 키워드 필터 후 제목 추출  
- `lastmod` 태그에서 날짜 추출

In [ ]:
def slug_to_title(slug):
    """URL 슬러그 → 읽을 수 있는 제목 변환 (하이픈 제거, Title Case)"""
    return slug.replace('-', ' ').title()


def fetch_sitemap_urls(sitemap_url):
    """
    개별 사이트맵 XML 파일 파싱
    - <url><loc>: 기사 URL
    - <url><lastmod>: 최종 수정 날짜
    - /news/, /markets/, /features/ 경로만 필터링 (광고 등 제외)
    """
    res = requests.get(sitemap_url, headers=HEADERS, timeout=20)
    res.raise_for_status()
    root = ET.fromstring(res.content)
    urls = root.findall('sm:url', NS)

    rows = []
    for u in urls:
        loc  = u.findtext('sm:loc',     namespaces=NS) or ''
        mod  = u.findtext('sm:lastmod', namespaces=NS) or ''
        date = mod[:10] if mod else ''

        # 분석 기간 필터
        if date and date < START_DATE:
            continue

        # 기사 경로만 포함
        if not any(seg in loc for seg in ['/news/', '/markets/', '/features/']):
            continue

        # URL에 키워드가 포함된 기사만 수집
        if not any(kw in loc.lower() for kw in CT_KEYWORDS):
            continue

        # URL 마지막 세그먼트를 제목으로 변환
        slug  = loc.rstrip('/').split('/')[-1]
        title = slug_to_title(slug)

        rows.append({
            'Date':   date,
            'title':  title,
            'text':   '',  # 본문은 별도 요청 필요 (스크래핑 금지)
            'url':    loc,
            'source': 'cointelegraph'
        })
    return rows

In [ ]:
# CoinTelegraph 사이트맵 수집 실행
print('사이트맵 인덱스 로딩...')
res = requests.get('https://cointelegraph.com/sitemap.xml', headers=HEADERS, timeout=15)
root = ET.fromstring(res.content)

# post-N.xml 형태의 사이트맵 파일만 선택
sitemap_locs = [
    s.findtext('sm:loc', namespaces=NS)
    for s in root.findall('sm:sitemap', NS)
    if 'post-' in (s.findtext('sm:loc', namespaces=NS) or '')
]
print(f'총 {len(sitemap_locs)}개 사이트맵 파일 발견')

ct_rows = []
for i, url in enumerate(sitemap_locs, 1):
    try:
        rows = fetch_sitemap_urls(url)
        ct_rows.extend(rows)
        print(f'  [{i:02d}/{len(sitemap_locs)}] {url.split("/")[-1]}: {len(rows)}개 (누적 {len(ct_rows)}개)')
        time.sleep(0.5)
    except Exception as e:
        print(f'  [{i:02d}] 오류: {e}')
        time.sleep(2)

print(f'\nCoinTelegraph 총 수집: {len(ct_rows)}개')

In [ ]:
# Reddit + CoinTelegraph 병합 후 저장
os.makedirs(SAVE_DIR, exist_ok=True)

df_all = pd.DataFrame(reddit_rows + ct_rows)

# 제목 기준 중복 제거 (같은 기사 여러 키워드에서 수집된 경우)
df_all = df_all.drop_duplicates(subset=['title']).reset_index(drop=True)

# 분석 기간 필터 + 날짜 오름차순 정렬
df_all = df_all[df_all['Date'] >= START_DATE].sort_values('Date').reset_index(drop=True)

save_path = os.path.join(SAVE_DIR, 'news_raw.csv')
df_all.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {save_path}')
print(f'총 {len(df_all)}개 기사')
print(df_all['source'].value_counts())
df_all.head()